In [ ]:
import pybamm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
import dfols
import signal
from scipy.integrate import solve_ivp
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import savgol_filter
from scipy.signal import find_peaks
from scipy import interpolate, integrate
from stopit import threading_timeoutable as timeoutable
import os, sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
from batfuns import *
plt.rcParams = set_rc_params(plt.rcParams)
import winsound
from pybamm import exp, constants, Parameter
import pickle
eSOH_DIR = "../data/esoh_R/"
oCV_DIR = "../data/ocv/"
cyc_DIR = "../data/cycling/"
fig_DIR = "../figures/figures_paper/"
res_DIR = "../data/results_paper/"
resistance_DIR = "../data/resistance/"
%matplotlib widget

In [55]:
def load_cycling_data_ch(cell,eSOH_DIR,oCV_DIR,cyc_DIR,cyc_no):
    cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
    cycles = np.array(dfe_0['N'].astype('int'))+1
    # print(cell_no)
    cyc_data_raw1 = pd.read_csv(cyc_DIR+'cycling_data_cell_'+cell_no+'.csv')
    N1 = cycles[cyc_no]
    # print(N1)
    cyc_data_raw = cyc_data_raw1[ cyc_data_raw1['Cycle number'] == N1 ]
    cyc_data = cyc_data_raw.reset_index(drop=True)
    t_c1 = cyc_data['Time [s]']-cyc_data['Time [s]'][0]
    t_c1 = t_c1.values
    I_c1 = cyc_data['Current [mA]']/1000
    I_c1 = I_c1.values
    V_c1 = cyc_data['Voltage [V]']
    V_c1 = V_c1.values
    E_c1 = cyc_data["Expansion [mu m]"]
    E_c1 = E_c1.values
    N1 = cycles[cyc_no]+1
    # print(N1)
    cyc_data_raw = cyc_data_raw1[ cyc_data_raw1['Cycle number'] == N1 ]
    cyc_data = cyc_data_raw.reset_index(drop=True)
    t_c2 = cyc_data['Time [s]']-cyc_data['Time [s]'][0]
    t_c2 = t_c2.values
    t_c2 = t_c2 + t_c1[-1]
    I_c2 = cyc_data['Current [mA]']/1000
    I_c2 = I_c2.values
    V_c2 = cyc_data['Voltage [V]']
    V_c2 = V_c2.values
    E_c2 = cyc_data["Expansion [mu m]"]
    E_c2 = E_c2.values

    t = np.concatenate([t_c1,t_c2])
    t = t-t[0]
    I = np.concatenate([I_c1,I_c2])
    V = np.concatenate([V_c1,V_c2])
    E = np.concatenate([E_c1,E_c2])
    Q = integrate.cumtrapz(I,t, initial=0)/3600 #Ah
    
    return t,V,I,Q,E

In [ ]:
cell = 6
cyc_no = 2
cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
cycles = np.array(dfe_0['N'].astype('int'))+1
# print(cell_no)
cyc_data_raw1 = pd.read_csv(cyc_DIR+'cycling_data_cell_'+cell_no+'.csv')
N1 = cycles[cyc_no]
# print(N1)
cyc_data_raw = cyc_data_raw1[ cyc_data_raw1['Cycle number'] == N1 ]
cyc_data = cyc_data_raw.reset_index(drop=True)
t_c1 = cyc_data['Time [s]']#-cyc_data['Time [s]'][0]
t_c1 = t_c1.values
I_c1 = cyc_data['Current [mA]']/1000
I_c1 = I_c1.values
V_c1 = cyc_data['Voltage [V]']
V_c1 = V_c1.values
E_c1 = cyc_data["Expansion [mu m]"]
E_c1 = E_c1.values
N1 = cycles[cyc_no]+1
# print(N1)
cyc_data_raw = cyc_data_raw1[ cyc_data_raw1['Cycle number'] == N1 ]
cyc_data = cyc_data_raw.reset_index(drop=True)
t_c2 = cyc_data['Time [s]']
t_c2 = t_c2.values
t_c2 = t_c2
I_c2 = cyc_data['Current [mA]']/1000
I_c2 = I_c2.values
V_c2 = cyc_data['Voltage [V]']
V_c2 = V_c2.values
E_c2 = cyc_data["Expansion [mu m]"]
E_c2 = E_c2.values

t = np.concatenate([t_c1,t_c2])
I = np.concatenate([I_c1,I_c2])
V = np.concatenate([V_c1,V_c2])
E = np.concatenate([E_c1,E_c2])
Q = integrate.cumtrapz(I,t, initial=0)/3600 #Ah
fig, ax = plt.subplots(1,1)
ax.plot(t,V)

In [ ]:
spm = pybamm.lithium_ion.SPM(
    {
        "stress-induced diffusion": "false",
    }
)
param=spm.param
parameter_values = get_parameter_values()

In [56]:
def model_error(data_time,data_current,data_voltage,parameter_values,SOC_0=0,plot_bool=False):
    # t_int = np.arange(0,data_time[-1],1)
    t_int = data_time
    f_I_int = interpolate.interp1d(data_time,-data_current)
    f_V_int = interpolate.interp1d(data_time,data_voltage)

    I_int = f_I_int(t_int)
    V_int = f_V_int(t_int)
    timescale = parameter_values.evaluate(spm.timescale)
    current_interpolant = pybamm.Interpolant(
    #  drive_cycle[:, 0], drive_cycle[:, 1], timescale * pybamm.t
    t_int, I_int, timescale * pybamm.t
    )

    pybamm.set_logging_level("NOTICE")
    parameter_values["Current function [A]"] = current_interpolant
    # var_pts=var_pts,
    sim = pybamm.Simulation(spm, 
                            parameter_values=parameter_values, 
                            solver=pybamm.CasadiSolver(
                                mode="safe", 
                                rtol=1e-6, 
                                atol=1e-6,
                                dt_max=10,
                                )
                            )
    # sim.solve(initial_soc = SOC_0)
    sim.solve(t_eval=t_int,initial_soc = SOC_0)
    solution = sim.solution
    t = solution["Time [s]"].entries
    V_sim = solution["Terminal voltage [V]"].entries
    I_sim = solution["Current [A]"].entries
    # Q = -solution['Discharge capacity [A.h]'].entries
    # f_I_sim = interpolate.interp1d(t,I,bounds_error=False,fill_value="extrapolate")
    # f_V_sim = interpolate.interp1d(t,V,bounds_error=False,fill_value="extrapolate")
    # I_sim = f_I_sim(t_int)
    # V_sim = f_V_sim(t_int)
    I_int = f_I_int(t)
    V_int = f_V_int(t)
    rmse = pybamm.rmse(V_int,V_sim)
    if plot_bool:
        fig, ax1 = plt.subplots(1, 1, figsize=(5, 4), sharex=True)
        ax1.plot(t, V_int, 'k')
        ax1.plot(t, V_sim, 'b--')
        ax1.text(0.7,0.2,f'RMSE: {rmse*1e3:0.1f} mV',transform=ax1.transAxes)
        # ax1.plot(t1, V1, 'r-.')
        ax1.set_ylabel("Voltage [V]")
        # ax1.set_title("Cell 01 Fresh HPPC")
        ax1.set_ylim([3,4.25])
        ax1.set_xlabel("Time [s]")
        # ax1.set_xlim([20370,20470])
        ax1.legend(['data','sim'])
        fig.tight_layout()
        plt.show()
        # fig.savefig(fig_DIR+str(int(10000*random.random()))+".png")
    return rmse

In [57]:
t_c5,V_c5,I_c5,Q_c5,E_c5 = load_cycling_data_ch(3,eSOH_DIR,oCV_DIR,cyc_DIR,0)
t_1p5c,V_1p5c,I_1p5c,Q_1p5c,E_1p5c = load_cycling_data_ch(6,eSOH_DIR,oCV_DIR,cyc_DIR,0)
t_2c,V_2c,I_2c,Q_2c,E_2c = load_cycling_data_ch(9,eSOH_DIR,oCV_DIR,cyc_DIR,0)

In [58]:
T = 273.15+45
E_r = 39570.0
R = 8.314
np.exp(E_r / R * (1 / 298.15 - 1 / T))

2.7278245079582475

In [59]:
cell = 6
cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
eps_n_data,eps_p_data,c_rate_c,c_rate_d,dis_set,Temp,SOC_0 = init_exp(cell_no,dfe_0,spm,parameter_values)
parameter_values = get_parameter_values()
def update_parameters(x):
    parameter_values = get_parameter_values()
    parameter_values.update(
        {
            "Negative electrode active material volume fraction": eps_n_data,
            "Positive electrode active material volume fraction": eps_p_data,
            "Initial temperature [K]": 273.15+Temp,
            "Ambient temperature [K]": 273.15+Temp,
            "Initial inner SEI thickness [m]": 0e-09,
            "Initial outer SEI thickness [m]": 5e-09,
            "SEI resistivity [Ohm.m]": 30000.0,
            "Upper voltage cut-off [V]":4.22,
        },
        check_already_exists=False,
    )
    parameter_values.update(
        {
            "Positive electrode diffusion coefficient activation energy [J.mol-1]": x[0]*10000.0*0,
            "Negative electrode diffusion coefficient activation energy [J.mol-1]": x[1]*10000.0*0,
            "Positive electrode reference exchange-current density activation energy [J.mol-1]": x[2]*10000.0*0,
            "Negative electrode reference exchange-current density activation energy [J.mol-1]": x[3]*10000.0*0,
            
            "Positive electrode diffusion coefficient [m2.s-1]": x[0]*8e-15,# *0.93,
            "Negative electrode diffusion coefficient [m2.s-1]": x[1]*8e-14,# *30.2,
            "Positive electrode reference exchange-current density [A.m-2(m3.mol)1.5]": x[2]*3.377e-06,#*3.3,
            "Negative electrode reference exchange-current density [A.m-2(m3.mol)1.5]":	x[3]*3.183e-06,#*0.62,
            
            # "Positive electrode OCP entropic change [V.K-1]": 0,
            # "Negative electrode OCP entropic change [V.K-1]": 0,
        },
        check_already_exists=False,
    )
    return parameter_values
x = np.array([1.0,1.0,1.0,1.0])
parameter_values = update_parameters(x)
plot_bool = True
rmse1 = model_error(t_c5,I_c5,V_c5,parameter_values,SOC_0=0.005,plot_bool=plot_bool)
rmse2 = model_error(t_1p5c,I_1p5c,V_1p5c,parameter_values,SOC_0=0,plot_bool=plot_bool)
rmse3 = model_error(t_2c,I_2c,V_2c,parameter_values,SOC_0=0,plot_bool=plot_bool)

RuntimeError: .../casadi/core/interpolant.cpp:68: Assertion "is_increasing(g)" failed:
Gridpoints must be strictly increasing

In [ ]:
dfgf

In [ ]:
def prediction_error(x):
    try:
        parameter_values = update_parameters(x)
        rmse1 = model_error(t_c5,I_c5,V_c5,parameter_values,SOC_0=0.005,plot_bool=False)
        rmse2 = model_error(t_1p5c,I_1p5c,V_1p5c,parameter_values,SOC_0=0,plot_bool=False)
        rmse3 = model_error(t_2c,I_2c,V_2c,parameter_values,SOC_0=0,plot_bool=False)
        # rmse5 = model_error(t_d1_hppc,I_d1_hppc,V_d1_hppc,parameter_values,SOC_0=1,plot_bool=False)
        out = rmse1 + rmse2 + rmse3 
        out = np.array([out])
        # out = rmse1 
        print(f"x={x}, norm={np.linalg.norm(out)}")
    # except pybamm.SolverError:
    except Exception as error:
        out = 5
        out = np.array([out])
        print(f"x={x}, norm={np.linalg.norm(out)}")
        print(error)
        return out
    return out
def train_model():
    timer = pybamm.Timer()
    x0 = np.array([1.0,1.0,1.0,1.0])
    lower = np.array([1e-1, 1e-1, 1e-1, 1e-1])
    upper = np.array([1e+1, 1e+1, 1e+1, 1e+1])
    dfo_opts = {
        "init.random_initial_directions":True,
        "init.run_in_parallel": True,
    }
    # soln_dfols = dfols.solve(prediction_error, x0,bounds=(lower, upper), rhoend=1e-3, user_params=dfo_opts)
    soln_dfols = dfols.solve(prediction_error, x0, rhoend=1e-3, user_params=dfo_opts)
    print(timer.time())
    return soln_dfols

In [ ]:
soln_dfols = train_model()
xsol = soln_dfols.x

In [ ]:
def plot_fit(x,plot_bool = True):
    parameter_values = update_parameters(x)
    print("RMSE")
    rmse = model_error(t_c5,I_c5,V_c5,parameter_values,SOC_0=0.005,plot_bool=plot_bool)
    print(f"C/5: {rmse}")
    rmse = model_error(t_1p5c,I_1p5c,V_1p5c,parameter_values,SOC_0=0,plot_bool=plot_bool)
    print(f"1.5C: {rmse}")
    rmse = model_error(t_2c,I_2c,V_2c,parameter_values,SOC_0=0,plot_bool=plot_bool)
    print(f"2C: {rmse}")

In [ ]:
xsol

In [ ]:
# # Ds, i0
# xsol = [1.12912078, 1.01642996, 0.70840774, 0.2680896]
# # Activation energies
# xsol = [-0.27280449, 13.43951307,  4.70920307, -1.83341506]

In [ ]:
plot_fit(xsol)